## Notebook 1: Data
Collate data into raw form of {OHLCV percentage changes + technical analysis of all 8 stocks on day n : best performing stock on day n + 1}

In [1]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
from datetime import date

In [2]:
tickers = ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'META', 'GOOGL', 'TSLA', 'AMD']

df = yf.download(tickers, start=date(2020, 5, 7), end=date.today())

all_features = []

for ticker in tickers:
    ohlcv = df.xs(ticker, level=1, axis=1).copy()
    ohlcv.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
    
    # Percent change windows
    ohlcv['ret_1d'] = ohlcv['Close'].pct_change(1)
    ohlcv['ret_5d'] = ohlcv['Close'].pct_change(5)
    ohlcv['ret_21d'] = ohlcv['Close'].pct_change(21)
    ohlcv['vol_chg'] = ohlcv['Volume'].pct_change(1)
    
    # Technical indicators
    ohlcv.ta.rsi(append=True)
    ohlcv.ta.macd(append=True)
    ohlcv.ta.bbands(append=True)
    ohlcv.ta.atr(append=True)
    ohlcv.ta.obv(append=True)
    
    # Prefix all indicator columns with ticker
    indicator_cols = [c for c in ohlcv.columns if c not in ['Open','High','Low','Close','Volume']]
    ohlcv = ohlcv[indicator_cols].add_prefix(f'{ticker}_') if not all(c.startswith(ticker) for c in indicator_cols) else ohlcv[indicator_cols]
    
    all_features.append(ohlcv)

features_df = pd.concat(all_features, axis=1)

# Label: best performing stock next day
close_df = df['Close']
daily_returns = close_df.pct_change(1)

shifted = daily_returns.shift(-1)
features_df['label'] = shifted.dropna(how='all').idxmax(axis=1)
features_df.dropna(inplace=True)


print(features_df.shape)
features_df.head()

[*********************100%***********************]  8 of 8 completed


(717, 121)


,NVDA_ret_1d,NVDA_ret_5d,NVDA_ret_21d,NVDA_vol_chg,NVDA_RSI_14,NVDA_MACD_12_26_9,NVDA_MACDh_12_26_9,NVDA_MACDs_12_26_9,NVDA_BBL_5_2.0_2.0,NVDA_BBM_5_2.0_2.0,...,AMD_MACDh_12_26_9,AMD_MACDs_12_26_9,AMD_BBL_5_2.0_2.0,AMD_BBM_5_2.0_2.0,AMD_BBU_5_2.0_2.0,AMD_BBB_5_2.0_2.0,AMD_BBP_5_2.0_2.0,AMD_ATRr_14,AMD_OBV,label
Date,,,,,,,,,,,,,,,,,,,,,
2023-06-26,-0.037362,-0.048252,0.330677,0.659468,55.545581,3.147669,-0.543935,3.691604,40.094202,42.510571,...,-3.833668,6.136798,103.264937,111.852,120.439064,15.354332,0.247178,5.402828,337727000.0,TSLA
2023-06-27,0.030616,-0.044101,0.102694,-0.222349,59.722761,2.961519,-0.584068,3.545587,40.138760,42.124474,...,-3.542029,5.251291,106.798749,110.144,113.489252,6.074323,0.536768,5.294769,397108600.0,TSLA
2023-06-28,-0.018125,-0.044791,0.055853,0.260646,56.249792,2.721426,-0.659329,3.380755,39.869702,41.739174,...,-3.235470,4.442423,107.192115,109.756,112.319884,4.671971,0.580737,5.213000,329574500.0,AMD
2023-06-29,-0.007175,-0.051203,0.017831,-0.346913,54.913164,2.478791,-0.721571,3.200362,39.935655,41.298917,...,-2.845933,3.730940,107.066447,109.864,112.661553,5.092757,0.745929,5.049214,387921700.0,NVDA
2023-06-30,0.036255,0.002203,0.118211,0.317029,60.042959,2.378415,-0.657558,3.035973,39.893281,41.317502,...,-2.316210,3.151887,106.046922,110.644,115.241079,8.309675,0.855226,4.934985,441300500.0,TSLA


In [3]:
## Save to csv
features_df.to_csv('../data/raw_master.csv')